In [1]:
#installs
!pip install -q transformers datasets evaluate accelerate scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [2]:
#imports
import random, numpy as np, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
Trainer, TrainingArguments, set_seed)
from datasets import load_dataset
import evaluate
import pandas as pd


In [3]:
#config
MODEL_NAME = "distilbert-base-uncased"
TASK = "sst2"
NUM_LABELS = 2
SEEDS = [1, 11, 21, 31, 42]

In [4]:
#helper function

def set_all_seeds(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
  set_seed(seed)

def preprocess_fn(examples):
  return tokenizer(examples["sentence"], truncation=True, padding="max_length", max_length=128)

def compute_metrics(p):
  preds = np.argmax(p.predictions, axis=1)
  return metric.compute(predictions=preds, references=p.label_ids)

In [5]:
#load dataset + tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
dataset = load_dataset("glue", TASK)
metric = evaluate.load("glue", TASK)

dataset = dataset.map(preprocess_fn, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [6]:
#baseline

seed = 42
set_all_seeds(seed)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

training_args = TrainingArguments(
  output_dir=f"./out/{TASK}_baseline",
  eval_strategy="epoch",
  save_strategy="epoch",
  learning_rate=2e-5,
  per_device_train_batch_size=32,
  per_device_eval_batch_size=64,
  num_train_epochs=3,
  weight_decay=0.01,
  warmup_ratio=0.06,
  load_best_model_at_end=True,
  metric_for_best_model="accuracy",
  seed=seed,
  fp16=True,
  report_to="none",
)

trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=dataset["train"],
  eval_dataset=dataset["validation"],
  tokenizer=tokenizer,
  compute_metrics=compute_metrics,
)

trainer.train()
results_baseline = trainer.evaluate()
print(results_baseline)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2896392300.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.185100,0.259007,0.905963
2,0.121900,0.312099,0.896789
3,0.080900,0.363265,0.896789


{'eval_loss': 0.25900736451148987, 'eval_accuracy': 0.9059633027522935, 'eval_runtime': 0.8529, 'eval_samples_per_second': 1022.422, 'eval_steps_per_second': 16.415, 'epoch': 3.0}


In [7]:
from google.colab import files
import shutil
import os

# zip all model outputs
!zip -r distilbert_sst2_models.zip ./out/

#download all models as one archive
files.download("distilbert_sst2_models.zip")


  adding: out/ (stored 0%)
  adding: out/sst2_baseline/ (stored 0%)
  adding: out/sst2_baseline/checkpoint-6315/ (stored 0%)
  adding: out/sst2_baseline/checkpoint-6315/optimizer.pt (deflated 29%)
  adding: out/sst2_baseline/checkpoint-6315/vocab.txt (deflated 53%)
  adding: out/sst2_baseline/checkpoint-6315/training_args.bin (deflated 53%)
  adding: out/sst2_baseline/checkpoint-6315/model.safetensors (deflated 8%)
  adding: out/sst2_baseline/checkpoint-6315/scaler.pt (deflated 64%)
  adding: out/sst2_baseline/checkpoint-6315/trainer_state.json (deflated 69%)
  adding: out/sst2_baseline/checkpoint-6315/tokenizer.json (deflated 71%)
  adding: out/sst2_baseline/checkpoint-6315/tokenizer_config.json (deflated 75%)
  adding: out/sst2_baseline/checkpoint-6315/scheduler.pt (deflated 61%)
  adding: out/sst2_baseline/checkpoint-6315/special_tokens_map.json (deflated 42%)
  adding: out/sst2_baseline/checkpoint-6315/config.json (deflated 45%)
  adding: out/sst2_baseline/checkpoint-6315/rng_state

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
#perturbation 1: random inits with different seeds
all_results = []
for seed in SEEDS:
  set_all_seeds(seed)
  model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
  args = TrainingArguments(
    output_dir=f"./out/{TASK}_seed{seed}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.06,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    seed=seed,
    fp16=True,
    report_to="none",
)

  trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    )

  trainer.train()
  res = trainer.evaluate()
  res['seed'] = seed
  all_results.append(res)

pd.DataFrame(all_results).to_csv("sst2_seed_variance.csv", index=False)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-818802196.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.183300,0.255369,0.903670
2,0.117800,0.331445,0.893349
3,0.080000,0.378830,0.897936


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-818802196.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.181100,0.279459,0.901376
2,0.112700,0.313722,0.901376
3,0.081800,0.345256,0.902523


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-818802196.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.182700,0.294970,0.889908
2,0.125700,0.304551,0.909404
3,0.085400,0.350426,0.905963


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-818802196.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.180700,0.321366,0.892202
2,0.118300,0.261234,0.912844
3,0.081600,0.325703,0.911697


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-818802196.py:23: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.185100,0.259007,0.905963
2,0.121900,0.312099,0.896789
3,0.080900,0.363265,0.896789


In [12]:
from google.colab import files
import shutil
import os

files.download("sst2_seed_variance.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
#perturbation 2: subset of data

fractions = [0.5, 0.25]
subset_results = []
for frac in fractions:
  subset = dataset["train"].train_test_split(test_size=1-frac, seed=42)["train"]
  model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
  args = TrainingArguments(
    output_dir = f"./out/{TASK}_subset{int(frac*100)}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.06,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    seed=seed,
    fp16=True,
    report_to="none",
  )

  trainer = Trainer(
    model=model,
    args=args,
    train_dataset=subset,
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
  )

  trainer.train()
  res = trainer.evaluate()
  res['subset_frac'] = frac
  subset_results.append(res)

pd.DataFrame(subset_results).to_csv("sst2_subset_variance.csv", index=False)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1545478663.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.241500,0.243943,0.902523
2,0.140400,0.298043,0.902523
3,0.094800,0.348591,0.901376


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1545478663.py:25: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.345900,0.286022,0.879587
2,0.165900,0.289554,0.889908
3,0.107400,0.330047,0.903670


In [15]:
files.download("sst2_subset_variance.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
#perturbation 3: different train/dev split

split_results = []
for i, split_seed in enumerate([7, 17, 27]):
  split_ds = dataset["train"].train_test_split(test_size=0.1, seed=split_seed)
  model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
  args = TrainingArguments(
    output_dir = f"./out/{TASK}_split{split_seed}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.06,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    seed=seed,
    fp16=True,
    report_to="none",
  )


trainer = Trainer(
  model=model,
  args=args,
  train_dataset=split_ds["train"],
  eval_dataset=split_ds["test"],
  tokenizer=tokenizer,
  compute_metrics=compute_metrics,
)

trainer.train()
res = trainer.evaluate()
res['split_seed'] = split_seed
split_results.append(res)

pd.DataFrame(split_results).to_csv("sst2_split_variance.csv", index=False)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-strea

Epoch,Training Loss,Validation Loss,Accuracy
1,0.200400,0.169299,0.938382
2,0.123800,0.175608,0.945954
3,0.084100,0.199792,0.948033


In [18]:
files.download("sst2_split_variance.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
#advGLUE
from datasets import load_dataset, Dataset
import pandas as pd
import json, glob, os, sys, traceback

def load_adv_sst2_from_hf():
    return load_dataset("AI-Secure/adv_glue", "adv_sst2", split="validation")

#load dataset
adv_dataset = None
try:
    print("Attempting to load AdvGLUE SST-2 from Hugging Face (AI-Secure/adv_glue adv_sst2)...")
    adv_dataset = load_adv_sst2_from_hf()
    print("Loaded from HF. Rows:", len(adv_dataset))
except Exception as e:
    print("HF load failed:", str(e))

#print for understanding dataset
print("Columns:", adv_dataset.column_names)
print("First example:", adv_dataset[0])


from datasets import DatasetDict

text_col = None
for c in ["sentence", "text", "premise", "sentence1"]:
    if c in adv_dataset.column_names:
        text_col = c
        break
if text_col is None:
    # try to find first string column
    for c in adv_dataset.column_names:
        if adv_dataset.features[c].dtype == 'string':
            text_col = c
            break
if text_col is None:
    raise RuntimeError("Couldn't find a text column in AdvGLUE SST-2 dev dataset. Columns: " + str(adv_dataset.column_names))

print("Using text column:", text_col)

#tokenizer
def adv_preprocess(examples):
    return tokenizer(examples[text_col], truncation=True, padding="max_length", max_length=128)

adv_dataset = adv_dataset.map(adv_preprocess, batched=True)

#evaluate
try:
    adv_results = trainer.evaluate(eval_dataset=adv_dataset)
    print("AdvGLUE SST-2 Dev Evaluation Results:", adv_results)
except Exception as e:
    print("trainer.evaluate failed. Make sure 'trainer' is defined and points to a model you want to evaluate.")
    traceback.print_exc()
    raise

Attempting to load AdvGLUE SST-2 from Hugging Face (AI-Secure/adv_glue adv_sst2)...
Loaded from HF. Rows: 148
Columns: ['sentence', 'label', 'idx']
First example: {'sentence': "it 's an uneven treat that bores fun at the democratic exercise while also examining its significance for those who take part .", 'label': 1, 'idx': 0}
Using text column: sentence


Map:   0%|          | 0/148 [00:00<?, ? examples/s]

AdvGLUE SST-2 Dev Evaluation Results: {'eval_loss': 2.487266778945923, 'eval_accuracy': 0.2905405405405405, 'eval_runtime': 0.2505, 'eval_samples_per_second': 590.81, 'eval_steps_per_second': 11.976, 'epoch': 3.0}


In [11]:
print("GLUE dev accuracy:", results_baseline['eval_accuracy'])
print("AdvGLUE dev accuracy:", adv_results['eval_accuracy'])
drop = 100 * (results_baseline['eval_accuracy'] - adv_results['eval_accuracy'])
print(f"Adversarial performance drop: {drop:.2f}%")

GLUE dev accuracy: 0.9059633027522935
AdvGLUE dev accuracy: 0.2905405405405405
Adversarial performance drop: 61.54%


In [12]:
pd.DataFrame([{
    "seed": 42,
    "glue_accuracy": results_baseline["eval_accuracy"],
    "advglue_accuracy": adv_results["eval_accuracy"],
    "drop_percent": drop,
}]).to_csv("sst2_seed42_advglue_results.csv", index=False)


In [13]:
files.download("sst2_seed42_advglue_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>